# QuantAgent-RL — Agents Module Demo

This notebook demonstrates the full multi-agent LLM pipeline that produces the qualitative component of the RL state vector.

### System Architecture

```
START ──────────────────────────────────────────
       │                              │
 [MacroAgent]               [SectorAgent × N]
  FRED signals +              GICS sector news
  web search                  + earnings data
       │                              │
       └─────────── [CompanyAgent × M] ────────┘
                     EDGAR XBRL + MD&A text
                              │
                  [OrchestratorAgent]
                   Synthesize → MarketBrief
                              │
                  [MarketBriefEmbedder]
                   GPU sentence-transformer
                   → dense RL state vector
```

> **All cells run in `mock_mode=True`** — no Anthropic API key required.
> Set `ANTHROPIC_API_KEY` and `mock_mode=False` to run with real Claude calls.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print('Libraries loaded.')

## 0. Configuration

In [ ]:
from agents import AgentConfig, HuggingFaceConfig

hf_cfg = HuggingFaceConfig(
    load_in_4bit=True
)

cfg = AgentConfig(
    mock_mode=False,         # ← set False + ANTHROPIC_API_KEY for real calls
    enable_web_search=True,  # ← set True to augment with live news (live mode only)
    temperature=0.0,
    max_tokens=2048,
    llm_backend="huggingface",  # "claude" or "huggingface"
    huggingface=hf_cfg,
)

# Shared inputs used across all cells
AS_OF_DATE = '2023-09-30'
# TICKERS    = ['AAPL','MSFT','NVDA','AVGO','GOOGL','META',
#                'AMZN','JPM','BAC','JNJ','LLY','UNH',
#                'XOM','CVX','RTX','CAT','PG','KO','NEE','PLD']
TICKERS    = ['AAPL','NVDA']

# Representative FRED macro snapshot for the demo
MACRO_DATA = {
    'fed_funds_rate':     5.25,
    'cpi_yoy':            3.70,
    'vix':               17.50,
    'yield_curve_10y2y': -0.48,
    'hy_spread':         412.0,
    'unemployment_rate':   3.80,
    'consumer_sentiment': 67.90,
    'wti_crude_oil':      90.50,
}

print(f'Config: mock_mode={cfg.mock_mode}, model={cfg.model if cfg.llm_backend=="claude" else cfg.huggingface.model_name}')
print(f'Universe: {len(TICKERS)} tickers, as_of_date={AS_OF_DATE}')

## 1. Schemas — Structured Output Models

Each agent returns a typed dataclass with validated fields.
All validation happens in `__post_init__` — no Pydantic required.

In [ ]:
from agents.schemas import MacroBrief, SectorBrief, CompanyBrief, MarketBrief

# Neutral placeholders illustrate the schema structure
print('=== MacroBrief schema ===')
mb = MacroBrief.neutral()
print(json.dumps(mb.to_dict(), indent=2))

print('\n=== SectorBrief schema ===')
print(json.dumps(SectorBrief.neutral('Information Technology').to_dict(), indent=2))

print('\n=== CompanyBrief schema ===')
print(json.dumps(CompanyBrief.neutral('NVDA').to_dict(), indent=2))

## 2. MacroAgent

Analyzes FRED macro signals and synthesizes a `MacroBrief`.
In live mode, also issues date-bounded web searches for rate/inflation news.

In [ ]:
from agents import MacroAgent

macro_agent = MacroAgent(cfg)
macro_brief = macro_agent.run(macro_data=MACRO_DATA, as_of_date=AS_OF_DATE)

print('MacroBrief:')
print(json.dumps(macro_brief.to_dict(), indent=2))

In [ ]:
# Show heuristic fallback (used when LLM call fails)
# Demonstrates that the system degrades gracefully
heuristic_scenarios = [
    ('High stress', {'fed_funds_rate':5.5,'cpi_yoy':7.0,'vix':35.0,'yield_curve_10y2y':-0.8,'hy_spread':700.0}),
    ('Neutral',     {'fed_funds_rate':2.5,'cpi_yoy':2.5,'vix':18.0,'yield_curve_10y2y': 0.5,'hy_spread':350.0}),
    ('Easing',      {'fed_funds_rate':0.5,'cpi_yoy':1.2,'vix':13.0,'yield_curve_10y2y': 1.2,'hy_spread':280.0}),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (label, data) in zip(axes, heuristic_scenarios):
    hb = MacroBrief(**MacroAgent._heuristic_brief(data))
    scores = {
        'Sentiment':        (hb.overall_sentiment + 1) / 2,
        '1 − Recession':    1 - hb.recession_risk,
        '1 − Credit stress':1 - hb.credit_stress,
    }
    bars = ax.barh(list(scores.keys()), list(scores.values()),
                   color=['#2ca02c','#1f77b4','#ff7f0e'])
    ax.set_xlim(0, 1.05)
    ax.axvline(0.5, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(
        f'{label}\nRate: {hb.rate_environment}\n'
        f'Inflation: {hb.inflation_regime}\nYield curve: {hb.yield_curve_signal}',
        fontsize=9
    )

plt.suptitle('MacroAgent Heuristic Fallback — Three Macro Scenarios', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. SectorAgent

One agent per GICS sector. In live mode, issues sector-specific web searches
for earnings revision trends, analyst commentary, and thematic tailwinds.

In [ ]:
from agents import SectorAgent
from data.universe import TICKER_SECTOR_MAP

# Build sector map from universe
sector_map: dict[str, list[str]] = {}
for ticker in TICKERS:
    sector = TICKER_SECTOR_MAP.get(ticker, 'Unknown')
    sector_map.setdefault(sector, []).append(ticker)

print(f'Sectors in universe: {len(sector_map)}')
for s, tks in sorted(sector_map.items()):
    print(f'  {s:<30} {tks}')

In [ ]:
# Run SectorAgent for every sector
sector_briefs: dict[str, SectorBrief] = {}
for sector, tickers in sector_map.items():
    agent = SectorAgent(cfg, sector=sector, tickers=tickers)
    sector_briefs[sector] = agent.run(macro_brief=macro_brief, as_of_date=AS_OF_DATE)

print(f'Sector briefs produced: {len(sector_briefs)}')

# Visualize sector momentum scores
momentum_df = pd.Series(
    {s: sb.momentum_score for s, sb in sector_briefs.items()}
).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#d62728' if v < 0 else '#2ca02c' for v in momentum_df.values]
momentum_df.plot.barh(ax=ax1, color=colors, alpha=0.8)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title('Sector Momentum Score\n(−1 = strong headwinds, +1 = strong tailwinds)')
ax1.set_xlabel('Momentum Score')

# Valuation and revision signals
val_map = {'cheap': 1, 'fair': 0, 'stretched': -1}
rev_map = {'upgrades': 1, 'neutral': 0, 'downgrades': -1}
sectors = list(sector_briefs.keys())
val_scores = [val_map[sector_briefs[s].valuation_signal] for s in sectors]
rev_scores = [rev_map[sector_briefs[s].earnings_revision_trend] for s in sectors]

x = np.arange(len(sectors))
ax2.bar(x - 0.2, val_scores, 0.35, label='Valuation (cheap=+1, stretched=−1)',
        color='#1f77b4', alpha=0.8)
ax2.bar(x + 0.2, rev_scores, 0.35, label='EPS Revisions (upgrades=+1)',
        color='#ff7f0e', alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels([s.replace(' ', '\n') for s in sectors], fontsize=7)
ax2.set_title('Valuation & Earnings Revision Signals by Sector')
ax2.set_ylabel('Score')
ax2.legend(fontsize=8)

plt.suptitle(f'SectorAgent Outputs — {AS_OF_DATE}', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. CompanyAgent

Analyzes each stock using EDGAR XBRL financials and MD&A text.
Produces a `CompanyBrief` with revenue trend, margin trend, balance-sheet
quality, and a composite fundamental score in [−1, +1].

In [ ]:
from agents import CompanyAgent

company_agent  = CompanyAgent(cfg)
ticker_sector  = {t: s for s, tks in sector_map.items() for t in tks}
company_briefs: dict[str, CompanyBrief] = {}

for ticker in TICKERS:
    sector = ticker_sector.get(ticker, '')
    sb     = sector_briefs.get(sector)
    company_briefs[ticker] = company_agent.run(
        ticker=ticker, xbrl_facts={}, mda_text='',
        sector_brief=sb, as_of_date=AS_OF_DATE,
    )

print(f'Company briefs produced: {len(company_briefs)}')
scores = {t: cb.fundamental_score for t, cb in company_briefs.items()}
print('\nFundamental scores:')
for t, s in sorted(scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(abs(s) * 10) if s > 0 else '░' * int(abs(s) * 10)
    print(f'  {t:<6} {s:+.2f}  {bar}')

In [ ]:
# Fundamental score breakdown by sector
fund_df = pd.DataFrame({
    'ticker':            list(company_briefs.keys()),
    'sector':            [ticker_sector.get(t,'') for t in company_briefs],
    'fundamental_score': [cb.fundamental_score for cb in company_briefs.values()],
    'revenue_trend':     [cb.revenue_growth_trend for cb in company_briefs.values()],
    'margin_trend':      [cb.margin_trend for cb in company_briefs.values()],
    'bs_quality':        [cb.balance_sheet_quality for cb in company_briefs.values()],
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sorted fundamental scores
fund_sorted = fund_df.sort_values('fundamental_score')
bar_colors  = ['#d62728' if v < 0 else '#2ca02c' for v in fund_sorted['fundamental_score']]
ax1.barh(fund_sorted['ticker'], fund_sorted['fundamental_score'],
         color=bar_colors, alpha=0.85)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title('CompanyAgent — Fundamental Scores')
ax1.set_xlabel('Score (−1 = very negative, +1 = very positive)')

# Revenue trend distribution
trend_counts = fund_df['revenue_trend'].value_counts()
trend_palette = {
    'accelerating': '#2ca02c', 'stable': '#1f77b4',
    'decelerating': '#ff7f0e', 'negative': '#d62728'
}
colors_rev = [trend_palette.get(k, 'gray') for k in trend_counts.index]
ax2.bar(trend_counts.index, trend_counts.values, color=colors_rev, alpha=0.85)
ax2.set_title('Revenue Growth Trend Distribution')
ax2.set_ylabel('Number of companies')

plt.suptitle(f'CompanyAgent Outputs — {AS_OF_DATE}', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. OrchestratorAgent — MarketBrief Synthesis

Synthesizes all sub-analyses into a unified `MarketBrief`.
The orchestrator resolves contradictions and weights signals by conviction.

In [ ]:
from agents import OrchestratorAgent

orchestrator = OrchestratorAgent(cfg)
market_brief = orchestrator.run(
    as_of_date=AS_OF_DATE,
    tickers=TICKERS,
    macro_brief=macro_brief,
    sector_briefs=sector_briefs,
    company_briefs=company_briefs,
)

print('=== MarketBrief ===')
print(json.dumps(market_brief.to_dict(), indent=2))

In [ ]:
# MarketBrief dashboard
fig = plt.figure(figsize=(14, 9))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# 1. Regime gauge
ax1 = fig.add_subplot(gs[0, 0])
regime_scores = {'risk_off': 0, 'transitional': 0.5, 'risk_on': 1}
stance_scores = {'defensive': 0, 'neutral': 0.5, 'aggressive': 1}
r = regime_scores.get(market_brief.macro_regime, 0.5)
s = stance_scores.get(market_brief.portfolio_stance, 0.5)
ax1.barh(['Regime', 'Stance', 'Conviction'],
         [r, s, market_brief.conviction_score],
         color=['#1f77b4','#ff7f0e','#2ca02c'], alpha=0.8)
ax1.set_xlim(0, 1)
ax1.axvline(0.5, color='black', linewidth=0.5, linestyle='--')
ax1.set_title(f'Regime: {market_brief.macro_regime}\nStance: {market_brief.portfolio_stance}')

# 2. Sector tilts
ax2 = fig.add_subplot(gs[0, 1:3])
tilts = pd.Series(market_brief.sector_tilts).sort_values()
tilt_colors = ['#d62728' if v < 0 else '#2ca02c' for v in tilts]
tilts.plot.barh(ax=ax2, color=tilt_colors, alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Orchestrator Sector Tilts')
ax2.set_xlabel('Tilt (−1 = max underweight, +1 = max overweight)')

# 3. Top overweights vs underweights
ax3 = fig.add_subplot(gs[1, 0])
over_scores  = [company_briefs[t].fundamental_score for t in market_brief.top_overweights if t in company_briefs]
under_scores = [company_briefs[t].fundamental_score for t in market_brief.top_underweights if t in company_briefs]
ax3.barh(market_brief.top_overweights[:len(over_scores)],  over_scores,  color='#2ca02c', alpha=0.8, label='Overweight')
ax3.barh(market_brief.top_underweights[:len(under_scores)], under_scores, color='#d62728', alpha=0.8, label='Underweight')
ax3.axvline(0, color='black', linewidth=0.8)
ax3.set_title('Top Positions')
ax3.legend(fontsize=8)

# 4. Key themes wordcloud-style bar
ax4 = fig.add_subplot(gs[1, 1:3])
themes = market_brief.key_themes
y_pos  = np.arange(len(themes))
ax4.barh(y_pos, np.ones(len(themes)), color=plt.cm.tab10.colors[:len(themes)], alpha=0.7)
for i, theme in enumerate(themes):
    ax4.text(0.02, i, theme, va='center', fontsize=8.5)
ax4.set_yticks([])
ax4.set_xticks([])
ax4.set_title('Key Investment Themes')

plt.suptitle(f'MarketBrief Dashboard — {market_brief.as_of_date}', fontweight='bold', fontsize=13)
plt.show()

## 6. LangGraph StateGraph

In production, all agents run inside a `StateGraph` that fans out macro
and sector nodes in parallel before the company and orchestrator nodes.
A sequential fallback is used when `langgraph` is not installed.

In [ ]:
from agents.orchestrator import build_agent_graph, GraphState

graph = build_agent_graph(cfg)
print(f'Graph type: {type(graph).__name__}')

state = GraphState(
    as_of_date=AS_OF_DATE,
    tickers=TICKERS,
    sector_map=sector_map,
    macro_data=MACRO_DATA,
    xbrl_data={t: {} for t in TICKERS},
    mda_data={t: '' for t in TICKERS},
    macro_brief=None, sector_briefs={}, company_briefs={},
    market_brief=None, errors=[],
)

result = graph.invoke(state)
final_brief = result['market_brief']

print(f'\nGraph output:')
print(f'  Company briefs:  {len(result["company_briefs"])} tickers')
print(f'  Sector briefs:   {len(result["sector_briefs"])} sectors')
print(f'  Macro regime:    {final_brief.macro_regime}')
print(f'  Portfolio stance:{final_brief.portfolio_stance}')
print(f'  Conviction:      {final_brief.conviction_score:.2f}')
print(f'  Errors:          {result["errors"]}')

## 7. MarketBriefEmbedder — GPU-Accelerated Encoding

Converts the qualitative `MarketBrief` into a dense numeric vector that
the RL environment appends to the quantitative feature matrix.

In [ ]:
from agents import MarketBriefEmbedder

# device=None auto-detects CUDA; falls back to CPU gracefully
embedder = MarketBriefEmbedder(
    model_name=cfg.embedding_model,
    device=cfg.embedding_device,
    normalize=True,
)

vec = embedder.encode(market_brief)
print(f'Embedding backend : {embedder._backend}')
print(f'Embedding device  : {embedder._device}')
print(f'Embedding dim     : {embedder.embedding_dim}')
print(f'Vector shape      : {vec.shape}')
print(f'L2 norm           : {np.linalg.norm(vec):.6f}  (should be ≈1.0)')

In [ ]:
# Simulate quarterly briefs across multiple quarters and encode as a time series
quarterly_dates = pd.date_range('2020-03-31', '2023-09-30', freq='QE')

# Generate synthetic briefs with varying signals to show time-series embedding
briefs_by_date: dict[str, MarketBrief] = {}
regimes = ['risk_off','risk_off','transitional','risk_on','risk_on',
           'transitional','risk_on','risk_on','transitional','risk_off',
           'risk_off','risk_off','transitional','risk_on','risk_on']
for date, regime in zip(quarterly_dates, regimes[:len(quarterly_dates)]):
    stance = {'risk_on':'aggressive','risk_off':'defensive','transitional':'neutral'}[regime]
    briefs_by_date[date.strftime('%Y-%m-%d')] = MarketBrief(
        as_of_date=date.strftime('%Y-%m-%d'),
        macro_regime=regime,
        portfolio_stance=stance,
        conviction_score={'risk_on':0.7,'risk_off':0.8,'transitional':0.4}[regime],
        key_themes=[f'{regime} environment', 'AI capex cycle', 'Rate path uncertainty'],
        top_overweights=['NVDA','MSFT','LLY'] if regime=='risk_on' else ['KO','JNJ','PG'],
        top_underweights=['NEE','PLD'] if regime=='risk_on' else ['TSLA','AMZN'],
    )

embedding_df = embedder.encode_quarterly(briefs_by_date)
print(f'Quarterly embedding matrix: {embedding_df.shape}')
print(f'Date range: {embedding_df.index[0].date()} → {embedding_df.index[-1].date()}')

In [ ]:
# Visualize embedding dimensions over time (first 20 dims for readability)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Heatmap of embedding values
n_show = min(20, embedding_df.shape[1])
emb_show = embedding_df.iloc[:, :n_show]
sns.heatmap(
    emb_show.T,
    ax=ax1,
    cmap='RdYlGn',
    center=0,
    cbar_kws={'label': 'Embedding value'},
    xticklabels=[d.to_period('Q').strftime('%Y-Q%q') for d in embedding_df.index],
    yticklabels=[f'dim_{i}' for i in range(n_show)],
    linewidths=0.2,
)
ax1.set_title(f'MarketBrief Embedding — First {n_show} Dimensions Over Time')
plt.xticks(rotation=45, ha='right', fontsize=7)

# Pairwise cosine similarity between consecutive quarters
mats = embedding_df.values
cos_sim = [
    float(np.dot(mats[i], mats[i+1])) / (
        np.linalg.norm(mats[i]) * np.linalg.norm(mats[i+1]) + 1e-9
    )
    for i in range(len(mats)-1)
]
ax2.plot(embedding_df.index[1:], cos_sim, marker='o', color='steelblue', linewidth=1.2)
ax2.axhline(0.8, color='red', linestyle='--', linewidth=0.8, label='High similarity threshold')
ax2.set_title('Consecutive-Quarter Cosine Similarity of MarketBrief Embeddings')
ax2.set_ylabel('Cosine Similarity')
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'Mean consecutive similarity: {np.mean(cos_sim):.4f}')
print(f'Min consecutive similarity:  {np.min(cos_sim):.4f}  (regime transitions show low similarity)')

In [ ]:
# Benchmark encoding throughput
bench = embedder.benchmark(n_samples=100)
print('Embedding benchmark:')
for k, v in bench.items():
    print(f'  {k:<20}: {v}')

print()
print('GPU acceleration notes:')
print('  When sentence-transformers is installed and a CUDA GPU is available:')
print('  - Batch encoding (encode_quarterly) is ~10–25× faster than CPU')
print('  - Single encode() is bottlenecked by kernel launch overhead')
print('  - For 40+ quarterly briefs per fold, GPU encoding is strongly recommended')

## 8. AgentPipeline — Walk-Forward Execution

The `AgentPipeline` runs the graph for every quarter-end date in a
walk-forward fold, caches results to disk, and returns an `AgentBundle`.

In [ ]:
from agents.pipeline import AgentPipeline, AgentBundle
import tempfile
from pathlib import Path

# Simulate a minimal WalkForwardFold for demo purposes
from dataclasses import dataclass, field
import pandas as pd
import numpy as np

@dataclass
class MockFold:
    fold_idx: int = 0
    train_dates: pd.DatetimeIndex = field(
        default_factory=lambda: pd.date_range('2021-03-31','2022-12-31',freq='QE')
    )
    test_dates: pd.DatetimeIndex = field(
        default_factory=lambda: pd.date_range('2023-03-31','2023-09-30',freq='QE')
    )
    tickers: list = field(default_factory=lambda: ['AAPL','MSFT','NVDA','JPM'])
    macro: pd.DataFrame = field(
        default_factory=lambda: pd.DataFrame(
            {'fed_funds_rate': np.linspace(0.25,5.25,1000),
             'cpi_yoy':        np.linspace(2.0,8.0,1000),
             'vix':            np.random.default_rng(0).uniform(15,35,1000)},
            index=pd.date_range('2021-01-01', periods=1000, freq='B')
        )
    )
    n_train_quarters: int = 2
    n_test_quarters:  int = 1
    train_start: pd.Timestamp = pd.Timestamp('2021-03-31')
    train_end:   pd.Timestamp = pd.Timestamp('2021-09-30')
    test_start:  pd.Timestamp = pd.Timestamp('2021-12-31')
    test_end:    pd.Timestamp = pd.Timestamp('2022-03-31')

mock_fold = MockFold()
print(f'Mock fold: {mock_fold.n_train_quarters} train + {mock_fold.n_test_quarters} test quarters')
print(f'Tickers: {mock_fold.tickers}')

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    pipeline = AgentPipeline(
        config=cfg,
        cache_dir=tmpdir,
    )
    bundle = pipeline.run_fold(fold=mock_fold, sec_metadata=None)

print(bundle)
print(f'\nQuarterly briefs: {bundle.n_quarters}')
print(f'Embedding shape : {bundle.embeddings.shape}')
print(f'Embedding dim   : {bundle.embedding_dim}')

# Sample brief
sample_date  = list(bundle.briefs.keys())[0]
sample_brief = bundle.briefs[sample_date]
print(f'\nSample brief ({sample_date}):')
print(f'  regime={sample_brief.macro_regime}, stance={sample_brief.portfolio_stance}')
print(f'  conviction={sample_brief.conviction_score:.2f}')

sample_vec = bundle.get_embedding(sample_date)
print(f'  embedding: {sample_vec.shape}, norm={np.linalg.norm(sample_vec):.4f}')

In [ ]:
# Show embedding similarity across quarters in the bundle
emb_mat = bundle.embeddings.values
n = len(emb_mat)
sim_mat = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_mat[i,j] = np.dot(emb_mat[i], emb_mat[j]) / (
            np.linalg.norm(emb_mat[i]) * np.linalg.norm(emb_mat[j]) + 1e-9
        )

dates_str = [d.strftime('%Y-Q%q') for d in bundle.embeddings.index]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    sim_mat,
    ax=ax,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=True, fmt='.2f', fontsize=8,
    xticklabels=dates_str,
    yticklabels=dates_str,
    linewidths=0.5,
)
ax.set_title('MarketBrief Embedding Cosine Similarity Matrix\n(AgentBundle — Fold 0)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Connecting to the RL State Vector

The `AgentBundle.embeddings` DataFrame is the qualitative component of
the RL state. The RL environment stacks it with the quantitative feature
matrix from `data.features` and the forecasting features from `forecasting.pipeline`.

In [ ]:
# Illustrate how the three state components combine
n_q     = bundle.n_quarters
n_tickers = len(mock_fold.tickers)

# Simulated shapes from other modules
quant_features_shape   = (n_q, n_tickers * 14)  # data module: ~14 feature groups × N assets
forecast_features_shape = (n_q, n_tickers * 5)  # forecasting: vol + regime + 3 FF betas
embedding_shape         = bundle.embeddings.shape  # agents: sentence embedding

total_dim = quant_features_shape[1] + forecast_features_shape[1] + embedding_shape[1]

print('RL State Vector Composition')
print('─' * 50)
print(f'  data module features  : {quant_features_shape[1]:>5} dims  ({n_q} quarters × {n_tickers} assets × ~14 groups)')
print(f'  forecasting features  : {forecast_features_shape[1]:>5} dims  ({n_q} quarters × {n_tickers} assets × 5 outputs)')
print(f'  agent embeddings      : {embedding_shape[1]:>5} dims  ({embedding_shape[0]} quarters × {embedding_shape[1]}-dim sentence vector)')
print('─' * 50)
print(f'  Total per-quarter dim : {total_dim:>5}')
print()
print('Architecture note:')
print('  At each quarterly step, the RL environment concatenates all three')
print('  feature blocks into a single flat vector of shape (total_dim,),')
print('  which is fed to the PPO policy network (2-layer MLP, 256 hidden units).')

## Next Steps

With the `agents` module producing quarterly `MarketBrief` embeddings,
the remaining modules are:

1. **`tax/`** — FIFO lot tracker + short/long-term capital gains calculator
2. **`rl/`** — Custom Gymnasium environment with:
   - State: `data` features + `forecasting` features + `agents` embeddings
   - Reward: Differential Sharpe Ratio − tax cost + tax-loss harvesting benefit
   - Action: Portfolio weight deltas (continuous, PPO via Stable-Baselines3)
3. **`backtest/`** — Walk-forward engine, Sharpe, Sortino, tax drag metrics